# ABC-Reject — Modèle MA(2) Hiérarchique

Implémentation de l'algorithme **ABC-Reject** (Vanilla ABC) pour le modèle de moyenne mobile d'ordre 2 (MA(2)) hiérarchique décrit dans la **Section 10** du supplément de :

> Clarté, G., Robert, C.P., Ryder, R.J., Stoehr, J. (2021). *Component-wise Approximate Bayesian Computation via Gibbs-like steps.* Biometrika.

L'objectif est de comparer les performances d'**ABC-Reject** avec celles d'**ABC-Gibbs** sur le dataset synthétique de la Section 10.2.

**Graine :** `SEED = 42`

---
## 0. Imports et paramètres globaux

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import invgamma, gaussian_kde
import time
import os

# -----------------------------------------------------------
# Graine — ne pas modifier pour reproductibilité
# -----------------------------------------------------------
SEED = 42

# -----------------------------------------------------------
# Paramètres du modèle — Section 10 du supplément
# -----------------------------------------------------------
T       = 100          # longueur de chaque série temporelle
n       = 5            # nombre de séries parallèles
alpha   = [1., 2., 3.] # hyperparamètre Dirichlet vrai
zeta    = [1., 1.]     # hyperparamètre Inverse-Gamma vrai

# -----------------------------------------------------------
# Budget computationnel — Section 10.2 du supplément
# N_TOT = 1.1e6 comme indiqué dans le papier
# Réduire à 50_000 pour un test rapide
# -----------------------------------------------------------
N_TOT  = 1_100_000
N_KEEP = 1_000

print(f"Modèle : n={n} séries, T={T} points, dim(θ)={3*n+5}")
print(f"Budget : N_tot={N_TOT:,}, N={N_KEEP}")

---
## 1. Modèle MA(2) hiérarchique

### Modèle

On observe $n$ séries temporelles $x_1, \ldots, x_n$ chacune de longueur $T$ :
$$x_j(t) = y_t + \mu_{j,1}\, y_{t-1} + \mu_{j,2}\, y_{t-2}, \qquad y_t \sim \mathcal{N}(0, \sigma_j^2)$$

### Prior hiérarchique

Pour $j = 1, \ldots, n$ :
- $(\beta_{j,1}, \beta_{j,2}, 1-\beta_{j,1}-\beta_{j,2}) \sim \text{Dir}(\alpha)$ 
- $\mu_j = (\beta_{j,1} - \beta_{j,2},\; 2(\beta_{j,1}+\beta_{j,2})-1)$ &emsp; *(garantit la stationnarité)*
- $\sigma_j^2 \sim \text{IG}(\varsigma_1, \varsigma_2)$

Hyperpriors : $\alpha \sim \text{Exp}(1)^{\otimes 3}$, $\;\varsigma \sim C_+^{\otimes 2}$

### Statistiques résumées et distance globale (Équation 4 du papier)

Pour chaque série $j$ :
$$w(x_j) = \sqrt{(\hat{\rho}_1(x_j) - \hat{\rho}_1(x_j^\star))^2 + (\hat{\rho}_2(x_j) - \hat{\rho}_2(x_j^\star))^2}$$
$$v(x_j) = \left|\hat{\sigma}^2_{\text{thin}}(x_j) - \hat{\sigma}^2_{\text{thin}}(x_j^\star)\right|$$
où $\hat{\sigma}^2_{\text{thin}}$ est la variance empirique sur les observations espacées de 3 pas (indépendantes dans MA(2)).

Distance globale utilisée par ABC-Reject :
$$\delta(x) = \sum_{j=1}^{n} \left(\frac{w(x_j)}{q_j} + \frac{v(x_j)}{q'_j}\right)$$
avec $q_j = Q_{0.1\%}(w_j)$ et $q'_j = Q_{0.1\%}(v_j)$ calculés par calibration.

In [ ]:
# -----------------------------------------------------------
# Simulation du modèle
# -----------------------------------------------------------

def sample_mu(alpha, rng):
    """Tire mu depuis Dirichlet(alpha) puis applique la transformation
    vers la région de stationnarité du MA(2)."""
    beta = rng.dirichlet(alpha)
    b1, b2 = beta[0], beta[1]
    return np.array([b1 - b2, 2*(b1 + b2) - 1])


def sample_sigma2(zeta, rng):
    """Tire sigma² depuis IG(zeta[0], zeta[1])."""
    return invgamma.rvs(zeta[0], scale=zeta[1], random_state=rng)


def simulate_ma2(mu, sigma2, T, rng):
    """Simule une série MA(2) de longueur T.
    x(t) = y_{t+2} + mu[0]*y_{t+1} + mu[1]*y_t,  y_t ~ N(0, sigma2)"""
    sig = np.sqrt(sigma2)
    y   = rng.normal(0, sig, T + 2)
    return np.array([y[t+2] + mu[0]*y[t+1] + mu[1]*y[t] for t in range(T)])


def generate_data(n, T, alpha, zeta, rng=None):
    """Génère n séries MA(2) hiérarchiques de longueur T.
    Retourne (x_list, mu_list, sigma2_list)."""
    if rng is None:
        rng = np.random.default_rng(42)
    x_list, mu_list, sigma2_list = [], [], []
    for _ in range(n):
        mu     = sample_mu(alpha, rng)
        sigma2 = sample_sigma2(zeta, rng)
        x      = simulate_ma2(mu, sigma2, T, rng)
        x_list.append(x)
        mu_list.append(mu)
        sigma2_list.append(sigma2)
    return x_list, mu_list, sigma2_list


def sample_hyperparams(rng):
    """Tire alpha ~ Exp(1)^3 et zeta ~ C+^2 depuis leurs hyperpriors.
    C+ : si U ~ Unif(0, pi/2) alors tan(U) ~ demi-Cauchy standard."""
    alpha_s = rng.exponential(scale=1.0, size=3)
    zeta_s  = np.array([
        np.tan(rng.uniform(0, np.pi / 2)),
        np.tan(rng.uniform(0, np.pi / 2))
    ])
    return alpha_s, zeta_s


# -----------------------------------------------------------
# Statistiques résumées et distances
# -----------------------------------------------------------

def autocorr(x, lag):
    """Autocorrélation empirique au lag donné."""
    xc = x - np.mean(x)
    return np.sum(xc[:len(x)-lag] * xc[lag:]) / (np.sum(xc**2) + 1e-12)


def compute_w(x_sim, x_obs_j):
    """Distance w basée sur les autocorrélations lags 1 et 2.
    Quasi-suffisante pour mu_j (Section 10 du supplément)."""
    d1 = autocorr(x_sim, 1) - autocorr(x_obs_j, 1)
    d2 = autocorr(x_sim, 2) - autocorr(x_obs_j, 2)
    return np.sqrt(d1**2 + d2**2)


def compute_v(x_sim, x_obs_j, T=T):
    """Distance v basée sur la variance des obs. sous-échantillonnées à pas 3.
    Justification : x(t) et x(t+3) sont indépendants dans MA(2).
    Quasi-suffisante pour sigma²_j."""
    M    = T // 3
    idx  = np.arange(1, M + 1) * 3 - 1
    var_sim = np.var(x_sim[idx])
    var_obs = np.var(x_obs_j[idx])
    return abs(var_sim - var_obs)


def delta_global(x_sim_list, x_obs_list, q_w, q_v):
    """Distance globale normalisée — Équation (4) du papier.
    delta(x) = sum_j [ w(x_j)/q_j + v(x_j)/q'_j ]"""
    total = 0.0
    for j in range(n):
        total += (compute_w(x_sim_list[j], x_obs_list[j]) / (q_w[j] + 1e-12)
                + compute_v(x_sim_list[j], x_obs_list[j]) / (q_v[j] + 1e-12))
    return total


print("Fonctions du modèle chargées.")

---
## 2. Données observées $x^\star$

In [ ]:
# Génération des données observées avec les vrais paramètres
rng_data = np.random.default_rng(SEED)
x_obs, mu_true, sigma2_true = generate_data(n, T, alpha, zeta, rng_data)

print("Vrais paramètres :")
for j in range(n):
    print(f"  Série {j+1} : mu = {np.array(mu_true[j]).round(3)},  sigma² = {sigma2_true[j]:.3f}")

# Visualisation
fig, axes = plt.subplots(n, 1, figsize=(12, 8), sharex=True)
for j in range(n):
    axes[j].plot(x_obs[j], color=f'C{j}', linewidth=0.8)
    axes[j].axhline(0, color='gray', linewidth=0.4, linestyle='--')
    axes[j].set_ylabel(f'$x_{j+1}$', fontsize=9)
    rho1 = autocorr(x_obs[j], 1)
    rho2 = autocorr(x_obs[j], 2)
    axes[j].set_title(
        rf'$\mu^\star={np.array(mu_true[j]).round(3)}$, '
        rf'$\sigma^2={sigma2_true[j]:.3f}$, '
        rf'$\hat{{\rho}}_1={rho1:.3f}$, $\hat{{\rho}}_2={rho2:.3f}$',
        fontsize=8
    )
axes[-1].set_xlabel('Temps $t$', fontsize=10)
fig.suptitle(r"Données observées $x^\star$ — $n=5$ séries MA(2), $T=100$",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_donnees_observees.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Calibration des quantiles de normalisation $q_j$, $q'_j$

$w$ et $v$ n'ont pas la même échelle. On les normalise par leurs quantiles à 0.1% estimés sur $N_{\text{cal}}$ simulations depuis le prior hiérarchique complet.

In [ ]:
N_CAL   = 10_000
rng_cal = np.random.default_rng(SEED + 1)

print(f"Calibration sur {N_CAL:,} simulations...")
t0 = time.perf_counter()

w_cal = np.zeros((N_CAL, n))
v_cal = np.zeros((N_CAL, n))

for i in range(N_CAL):
    alpha_i, zeta_i     = sample_hyperparams(rng_cal)
    x_sim_i, _, _       = generate_data(n, T, alpha_i, zeta_i, rng_cal)
    for j in range(n):
        w_cal[i, j] = compute_w(x_sim_i[j], x_obs[j])
        v_cal[i, j] = compute_v(x_sim_i[j], x_obs[j])

print(f"  Terminé en {time.perf_counter()-t0:.1f}s")

# Quantiles 0.1% (Section 10 du supplément)
q_w = np.quantile(w_cal, 0.001, axis=0)
q_v = np.quantile(v_cal, 0.001, axis=0)

print("Quantiles de normalisation (Q 0.1%) :")
for j in range(n):
    print(f"  Série {j+1} : q_w={q_w[j]:.5f},  q_v={q_v[j]:.5f}")

# Visualisation
fig, axes = plt.subplots(2, n, figsize=(14, 5), sharex='row')
for j in range(n):
    for row, (data, qval, label, color) in enumerate([
        (w_cal[:, j], q_w[j], f'w — série {j+1}', 'steelblue'),
        (v_cal[:, j], q_v[j], f'v — série {j+1}', 'darkorange')
    ]):
        axes[row, j].hist(data, bins=50, density=True, color=color, alpha=0.7)
        axes[row, j].axvline(qval, color='red', lw=2, linestyle='--',
                             label=f'Q0.1%={qval:.4f}')
        axes[row, j].set_title(label, fontsize=8)
        axes[row, j].legend(fontsize=7)
        if j == 0:
            axes[row, j].set_ylabel(['w(x_j)', 'v(x_j)'][row], fontsize=9)

fig.suptitle('Calibration — distributions de w et v sous le prior', fontsize=11)
plt.tight_layout()
plt.savefig('fig_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Algorithme ABC-Reject

**Algorithm 1 du papier :**

Pour $i = 1$ à $N_{\text{tot}}$ :
1. Tirer $\alpha^{(i)} \sim \text{Exp}(1)^3$, $\;\varsigma^{(i)} \sim C_+^2$
2. Simuler $n$ séries via `generate_data`
3. Calculer $\delta^{(i)} = \delta(x^{(i)}, x^\star)$

**Retourner** les $N$ paramètres avec les $N$ plus petites distances.

Le seuil effectif $\varepsilon^\star$ est le $N$-ième quantile des distances.

In [ ]:
def abc_reject(x_obs, q_w, q_v, N_tot, N_keep, seed,
               verbose=True, print_every=100_000):
    """
    ABC-Reject sur le modèle MA(2) hiérarchique.

    Paramètres
    ----------
    x_obs       : liste de n arrays (T,)
    q_w, q_v    : arrays (n,) — quantiles de normalisation
    N_tot       : budget total de simulations
    N_keep      : taille de l'échantillon posterior final
    seed        : graine

    Retourne
    --------
    dict : alphas, zetas, mus, sigma2s, distances,
           epsilon, cpu_time, n_simulations, acceptance_rate
    """
    rng_abc     = np.random.default_rng(seed)
    alphas_all  = np.zeros((N_tot, 3))
    zetas_all   = np.zeros((N_tot, 2))
    mus_all     = np.zeros((N_tot, n, 2))
    sigma2s_all = np.zeros((N_tot, n))
    dists_all   = np.full(N_tot, np.inf)

    t0 = time.perf_counter()

    for i in range(N_tot):
        # Étape 1 : hyperparamètres depuis leurs priors
        alpha_i, zeta_i = sample_hyperparams(rng_abc)

        # Étapes 2-3 : simuler n séries et calculer δ
        x_sim_i, mu_i, sigma2_i = generate_data(n, T, alpha_i, zeta_i, rng_abc)
        dists_all[i]   = delta_global(x_sim_i, x_obs, q_w, q_v)
        alphas_all[i]  = alpha_i
        zetas_all[i]   = zeta_i
        mus_all[i]     = np.array(mu_i)
        sigma2s_all[i] = np.array(sigma2_i)

        if verbose and (i + 1) % print_every == 0:
            el  = time.perf_counter() - t0
            eps = np.sort(dists_all[:i+1])[min(N_keep - 1, i)]
            print(f"  [{i+1:>9,}/{N_tot:,}]  "
                  f"ε={eps:.3f}  "
                  f"{(i+1)/el:.0f} sim/s  "
                  f"ETA={int((N_tot-i-1)/((i+1)/el))}s")

    cpu = time.perf_counter() - t0
    idx = np.argsort(dists_all)[:N_keep]
    eps = dists_all[idx[-1]]

    print(f"\n{'='*52}")
    print(f"  N_tot            = {N_tot:,}")
    print(f"  N acceptés       = {N_keep}")
    print(f"  ε effectif       = {eps:.4f}")
    print(f"  Taux acceptation = {N_keep/N_tot:.4%}")
    print(f"  Temps CPU        = {cpu:.1f}s  ({N_tot/cpu:.0f} sim/s)")
    print(f"{'='*52}")

    return dict(
        alphas          = alphas_all[idx],
        zetas           = zetas_all[idx],
        mus             = mus_all[idx],
        sigma2s         = sigma2s_all[idx],
        distances       = dists_all[idx],
        epsilon         = eps,
        cpu_time        = cpu,
        n_simulations   = N_tot,
        acceptance_rate = N_keep / N_tot
    )


print(f"Lancement ABC-Reject : {N_TOT:,} simulations...")
res = abc_reject(x_obs, q_w, q_v, N_TOT, N_KEEP, seed=SEED)

---
## 5. Diagnostic et visualisation

In [ ]:
# -------------------------------------------------------
# Échantillon prior (pour comparaison visuelle)
# -------------------------------------------------------
rng_pv = np.random.default_rng(SEED + 99)
mu1_prior = []
for _ in range(5000):
    a_i, z_i = sample_hyperparams(rng_pv)
    mu1_prior.append(sample_mu(a_i, rng_pv)[0])
mu1_prior = np.array(mu1_prior)

# -------------------------------------------------------
# Figure : posterior mu_1 (série 1) — Fig. 11 du papier
# -------------------------------------------------------
mu1_post  = res['mus'][:, 0, 0]
mu1_true  = float(mu_true[0][0])
xr        = np.linspace(-1.1, 1.1, 300)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(mu1_post, bins=40, density=True, color='steelblue', alpha=0.75,
        label='Posterior ABC-Reject')
ax.plot(xr, gaussian_kde(mu1_prior)(xr), 'k-', lw=1.8, label='Prior')
ax.axvline(mu1_true, color='red', lw=2.5,
           label=f'Vraie valeur = {mu1_true:.3f}')
ax.set_xlabel(r'$\mu_{1,1}$', fontsize=13)
ax.set_ylabel('Densité', fontsize=11)
ax.set_title(r'ABC-Reject — Posterior de $\mu_{1,1}$ vs prior'
             '\n(proche du prior → malédiction de la dimension)',
             fontsize=11)
ax.legend(fontsize=10)
ax.set_xlim(-1.1, 1.1)
plt.tight_layout()
plt.savefig('fig_posterior_mu1_abc.png', dpi=150, bbox_inches='tight')
plt.show()

# -------------------------------------------------------
# Toutes les marginales mu_{j,1}
# -------------------------------------------------------
fig, axes = plt.subplots(1, n, figsize=(15, 3.5), sharey=True)
for j in range(n):
    mu_j_post = res['mus'][:, j, 0]
    mu_j_true = float(mu_true[j][0])
    axes[j].hist(mu_j_post, bins=35, density=True,
                 color='steelblue', alpha=0.75)
    axes[j].plot(xr, gaussian_kde(mu1_prior)(xr), 'k-', lw=1.2, alpha=0.6)
    axes[j].axvline(mu_j_true, color='red', lw=2,
                    label=f'vrai={mu_j_true:.2f}')
    axes[j].set_title(f'Série {j+1}', fontsize=10)
    axes[j].set_xlabel(r'$\mu_{j,1}$', fontsize=10)
    axes[j].legend(fontsize=8)
    if j == 0:
        axes[j].set_ylabel('Densité', fontsize=10)

fig.suptitle(r'ABC-Reject — Posterior de $\mu_{j,1}$ pour les $n=5$ séries',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_posterior_all_series.png', dpi=150, bbox_inches='tight')
plt.show()

# -------------------------------------------------------
# Distribution des distances acceptées
# -------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(res['distances'], bins=50, color='steelblue', alpha=0.8, density=True)
ax.axvline(res['epsilon'], color='red', linestyle='--', lw=2,
           label=f"ε = {res['epsilon']:.3f}")
ax.set_xlabel(r'Distance $\delta$', fontsize=11)
ax.set_ylabel('Densité')
ax.set_title('Distances des paramètres acceptés')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('fig_distances.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Métriques de comparaison

In [ ]:
# -------------------------------------------------------
# Métrique 1 : biais et intervalles de crédibilité
# -------------------------------------------------------
print("MÉTRIQUE 1 — Estimation")
print("-" * 55)
for j in range(n):
    mu_j_true = np.array(mu_true[j])
    mu_j_post = res['mus'][:, j, :]
    mean_j    = np.mean(mu_j_post, axis=0)
    q025_j    = np.quantile(mu_j_post, 0.025, axis=0)
    q975_j    = np.quantile(mu_j_post, 0.975, axis=0)
    biais_j   = mean_j - mu_j_true
    print(f"Série {j+1} :")
    print(f"  mu*      = {mu_j_true.round(3)}")
    print(f"  E[mu]    = {mean_j.round(3)}   Biais = {biais_j.round(3)}")
    print(f"  IC95 mu1 = [{q025_j[0]:.3f}, {q975_j[0]:.3f}]  "
          f"mu2 = [{q025_j[1]:.3f}, {q975_j[1]:.3f}]")
    print()

In [ ]:
# -------------------------------------------------------
# Métrique 2 : distance posterior prédictive
# Métrique principale du papier (Section 10.2)
# Pour chaque theta^(i) accepté : simuler x_tilde ~ f(.|theta^(i))
# et mesurer delta(x_tilde, x_obs).
# Petit = bon (les theta acceptés génèrent des données proches de x*)
# -------------------------------------------------------

def posterior_predictive_dist(res, x_obs, q_w, q_v, seed, n_eval=200):
    rng_e = np.random.default_rng(seed)
    N     = min(n_eval, len(res['alphas']))
    idx   = np.random.default_rng(seed + 1).choice(len(res['alphas']), N, replace=False)
    dists = []
    for k in idx:
        x_pred, _, _ = generate_data(n, T, res['alphas'][k], res['zetas'][k], rng_e)
        dists.append(delta_global(x_pred, x_obs, q_w, q_v))
    dists = np.array(dists)
    return np.mean(dists), np.std(dists), dists


def prior_predictive_dist(n_samples, x_obs, q_w, q_v, seed):
    """Baseline : tirer depuis le prior sans critère d'acceptation."""
    rng_p = np.random.default_rng(seed)
    dists = []
    for _ in range(n_samples):
        a, z         = sample_hyperparams(rng_p)
        x_p, _, _    = generate_data(n, T, a, z, rng_p)
        dists.append(delta_global(x_p, x_obs, q_w, q_v))
    return np.mean(dists), np.std(dists)


print("Calcul des distances prédictives...")
mean_abc,   std_abc,   dists_abc   = posterior_predictive_dist(
    res, x_obs, q_w, q_v, seed=SEED + 100
)
mean_prior, std_prior = prior_predictive_dist(
    200, x_obs, q_w, q_v, seed=SEED + 200
)

print()
print("MÉTRIQUE 2 — Distance posterior prédictive")
print("-" * 52)
print(f"  Prior (baseline)   : {mean_prior:.1f} ± {std_prior:.1f}")
print(f"  ABC-Reject         : {mean_abc:.1f} ± {std_abc:.1f}")
print(f"  ABC-Gibbs (papier) : 274.1 ± 2.5")
print(f"  ABC-Reject (papier): 436.8 ± 1.6")
print()
print(f"  Amélioration ABC-Reject / prior : {(mean_prior-mean_abc)/mean_prior*100:.1f}%")
print(f"  Ratio Reject / Gibbs (papier)  : {436.8/274.1:.2f}×  (Gibbs est meilleur)")

# Visualisation
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(dists_abc, bins=35, density=True, color='steelblue', alpha=0.75,
        label=f'ABC-Reject : {mean_abc:.1f} ± {std_abc:.1f}')
ax.axvline(mean_abc,   color='steelblue',  lw=2.5)
ax.axvline(mean_prior, color='gray',       lw=1.5, linestyle=':',
           label=f'Prior : {mean_prior:.1f}')
ax.axvline(274.1,      color='darkorange', lw=2, linestyle='--',
           label='ABC-Gibbs (papier) : 274.1')
ax.axvline(436.8,      color='steelblue',  lw=2, linestyle='--',
           label='ABC-Reject (papier) : 436.8')
ax.set_xlabel(r'Distance prédictive $\delta(\tilde{x}, x^\star)$', fontsize=11)
ax.set_ylabel('Densité', fontsize=11)
ax.set_title('Distance posterior prédictive — plus petite = meilleur', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig_predictive_dist.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Tableau comparatif et sauvegarde

In [ ]:
print("=" * 58)
print(" TABLEAU COMPARATIF — Section 10.2 du supplément")
print("=" * 58)
print(f"{'Métrique':<38} {'Notre impl.':>10} {'Papier':>8}")
print("-" * 58)
print(f"{'N_tot simulations':<38} {N_TOT:>10,} {'1.1e6':>8}")
print(f"{'N acceptés':<38} {N_KEEP:>10} {'1000':>8}")
print(f"{'ε effectif':<38} {res['epsilon']:>10.4f} {'—':>8}")
print(f"{'Taux acceptation':<38} {res['acceptance_rate']:>9.4%} {'—':>8}")
print(f"{'Temps CPU (s)':<38} {res['cpu_time']:>10.1f} {'—':>8}")
print(f"{'Dist. prédictive ABC-Reject':<38} {mean_abc:>10.1f} {'436.8':>8}")
print(f"{'Dist. prédictive ABC-Gibbs':<38} {'[à mesurer]':>10} {'274.1':>8}")
print(f"{'Baseline prior':<38} {mean_prior:>10.1f} {'—':>8}")
print("=" * 58)

# Sauvegarde
np.savez(
    'resultats_abc_reject.npz',
    alphas          = res['alphas'],
    zetas           = res['zetas'],
    mus             = res['mus'],
    sigma2s         = res['sigma2s'],
    distances       = res['distances'],
    epsilon         = res['epsilon'],
    cpu_time        = res['cpu_time'],
    n_simulations   = N_TOT,
    acceptance_rate = res['acceptance_rate'],
    mean_ppd        = mean_abc,
    std_ppd         = std_abc,
    mean_prior      = mean_prior,
    SEED            = SEED,
    x_obs           = np.array(x_obs),
    mu_true         = np.array(mu_true),
    sigma2_true     = np.array(sigma2_true),
    q_w             = q_w,
    q_v             = q_v,
)
print("\nRésultats sauvegardés dans 'resultats_abc_reject.npz'")